In [1]:
# 먼저 환경을 만들고 할 것!
# conda create -n paddle_finetune python=3.10

In [1]:
!python --version

Python 3.12.6


In [4]:
import sys
print(sys.executable)
print(sys.version)

/home/j-i14a403/.conda/envs/paddle_finetune/bin/python
3.10.19 | packaged by conda-forge | (main, Jan 26 2026, 23:45:08) [GCC 14.3.0]


In [10]:
%pip install pillow

  Using cached pillow-12.1.0-cp310-cp310-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (8.8 kB)
Using cached pillow-12.1.0-cp310-cp310-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl (7.0 MB)
Note: you may need to restart the kernel to use updated packages.


In [13]:
from PIL import Image
import os

def resize_with_padding(
    img: Image.Image,
    target_size=(640, 640),
    pad_color=(0, 0, 0)
):
    """
    비율 유지 + 패딩으로 리사이즈
    """
    iw, ih = img.size
    tw, th = target_size

    scale = min(tw / iw, th / ih)
    nw, nh = int(iw * scale), int(ih * scale)

    img = img.resize((nw, nh), Image.BILINEAR)

    new_img = Image.new("RGB", target_size, pad_color)
    new_img.paste(img, ((tw - nw) // 2, (th - nh) // 2))

    return new_img


def process_dir(
    input_dir,
    output_dir,
    target_size=(640, 640)
):
    os.makedirs(output_dir, exist_ok=True)

    for fname in os.listdir(input_dir):
        if not fname.lower().endswith((".jpg", ".jpeg", ".png")):
            continue

        img_path = os.path.join(input_dir, fname)
        img = Image.open(img_path).convert("RGB")

        resized = resize_with_padding(img, target_size)
        resized.save(os.path.join(output_dir, fname))


# 예시 실행
process_dir(
    input_dir="./img/",
    output_dir="./img/",
    target_size=(640, 640)
)


In [14]:
pip install pillow tqdm

  Using cached tqdm-4.67.1-py3-none-any.whl.metadata (57 kB)
Using cached tqdm-4.67.1-py3-none-any.whl (78 kB)
Note: you may need to restart the kernel to use updated packages.


In [17]:
import json
import random
from pathlib import Path
from PIL import Image
from tqdm import tqdm

GENERATED_DIR = Path("./generated")
IMAGES_DIR = GENERATED_DIR / "images"
LABELS_DIR = GENERATED_DIR / "labels"

OUT_DIR = Path("./paddle_recog_dataset")
TRAIN_RATIO = 0.9
SEED = 42

DO_RESIZE = True
IMG_H = 32
MAX_W = 320
PAD = 2

TARGET_FIELDS = [
    "tracking_number",
    "region_code",
    "recipient_name",
    "recipient_address",
    "sender_name",
    "sender_address",
]

def sanitize_label(text: str) -> str:
    return str(text).replace("\t", " ").replace("\n", " ").strip()

def clamp_box(box, w, h):
    x1, y1, x2, y2 = box
    x1 = max(0, min(int(x1), w))
    x2 = max(0, min(int(x2), w))
    y1 = max(0, min(int(y1), h))
    y2 = max(0, min(int(y2), h))
    if x2 <= x1 or y2 <= y1:
        return None
    return (x1, y1, x2, y2)

def resize_for_recognition(img: Image.Image, imgH=32, maxW=320) -> Image.Image:
    w, h = img.size
    if h <= 0:
        return img
    ratio = w / h
    new_w = max(1, min(int(imgH * ratio), maxW))
    return img.resize((new_w, imgH), Image.BILINEAR)

def build_paddle_recognition_dataset():
    train_img_dir = OUT_DIR / "train" / "images"
    val_img_dir = OUT_DIR / "val" / "images"
    train_img_dir.mkdir(parents=True, exist_ok=True)
    val_img_dir.mkdir(parents=True, exist_ok=True)

    # ✅ labels.json(통합) 제외하고 개별 라벨만 사용
    label_files = sorted([p for p in LABELS_DIR.glob("*.json") if p.name != "labels.json"])
    if not label_files:
        raise FileNotFoundError(f"개별 라벨(.json) 파일이 없음: {LABELS_DIR}")

    random.seed(SEED)
    random.shuffle(label_files)

    split_idx = int(len(label_files) * TRAIN_RATIO)
    train_files = label_files[:split_idx]
    val_files = label_files[split_idx:]

    def process_split(split_name, files, out_img_dir):
        lines = []
        saved = 0
        missing_img = 0
        skipped_label = 0

        for jf in tqdm(files, desc=f"{split_name} processing"):
            with open(jf, "r", encoding="utf-8") as f:
                item = json.load(f)

            # ✅ 방어: 혹시 리스트(통합)면 스킵
            if isinstance(item, list):
                skipped_label += 1
                continue
            if not isinstance(item, dict):
                skipped_label += 1
                continue

            rel_img_path = item.get("image_path")
            if not rel_img_path:
                skipped_label += 1
                continue

            img_path = GENERATED_DIR / rel_img_path
            if not img_path.exists():
                missing_img += 1
                continue

            img = Image.open(img_path).convert("RGB")
            w, h = img.size
            stem = Path(rel_img_path).stem  # "00001"

            for field in item.get("fields", []):
                field_name = field.get("field_name")
                if field_name not in TARGET_FIELDS:
                    continue

                gt = sanitize_label(field.get("text", ""))
                bbox = field.get("bbox")
                if not gt or not bbox:
                    continue

                x1, y1, x2, y2 = bbox
                box = clamp_box((x1 - PAD, y1 - PAD, x2 + PAD, y2 + PAD), w, h)
                if box is None:
                    continue

                crop = img.crop(box)
                if DO_RESIZE:
                    crop = resize_for_recognition(crop, imgH=IMG_H, maxW=MAX_W)

                out_name = f"{stem}_{field_name}.jpg"
                out_path = out_img_dir / out_name
                crop.save(out_path, "JPEG", quality=95)

                rel_out = f"{split_name}/images/{out_name}"
                lines.append(f"{rel_out}\t{gt}")
                saved += 1

        return lines, saved, missing_img, skipped_label

    train_lines, train_saved, train_missing, train_skipped = process_split("train", train_files, train_img_dir)
    val_lines, val_saved, val_missing, val_skipped = process_split("val", val_files, val_img_dir)

    with open(OUT_DIR / "train" / "label.txt", "w", encoding="utf-8") as f:
        f.write("\n".join(train_lines) + ("\n" if train_lines else ""))
    with open(OUT_DIR / "val" / "label.txt", "w", encoding="utf-8") as f:
        f.write("\n".join(val_lines) + ("\n" if val_lines else ""))

    print("\n✅ PaddleOCR Recognition 데이터셋 생성 완료")
    print(f"- 개별 라벨 json 수: {len(label_files)}")
    print(f"- Train crop: {train_saved} / missing images: {train_missing} / skipped labels: {train_skipped}")
    print(f"- Val   crop: {val_saved} / missing images: {val_missing} / skipped labels: {val_skipped}")
    print(f"📁 Output: {OUT_DIR}")

build_paddle_recognition_dataset()


val processing: 100%|██████████| 3000/3000 [00:13<00:00, 219.47it/s]



✅ PaddleOCR Recognition 데이터셋 생성 완료
- 개별 라벨 json 수: 30000
- Train crop: 162000 / missing images: 0 / skipped labels: 0
- Val   crop: 18000 / missing images: 0 / skipped labels: 0
📁 Output: paddle_recog_dataset


In [18]:
from pathlib import Path

chars = set()
for p in [Path("./paddle_recog_dataset/train/label.txt"), Path("./paddle_recog_dataset/val/label.txt")]:
    for line in p.read_text(encoding="utf-8").splitlines():
        if "\t" not in line:
            continue
        _, txt = line.split("\t", 1)
        chars.update(list(txt.strip()))

out = Path("./autobox_dict.txt")
out.write_text("\n".join(sorted(chars)), encoding="utf-8")
print("dict size:", len(chars), "->", out)


dict size: 230 -> autobox_dict.txt
